<a href="https://colab.research.google.com/github/Adonimac/DAE_proeject_1/blob/main/Predicting_Flame_Heat_Output_in_Draconic_Species_A_Regression_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 150, 'font.size': 10,
                     'axes.titlesize': 11, 'axes.labelsize': 10})
COLORS = {'Dragon': '#c0392b', 'Wyvern': '#2980b9', 'Hydra': '#27ae60'}

In [ ]:
from google.colab import files
uploaded = files.upload()   # opens a file picker — select dragon_data.csv

import pandas as pd
df = pd.read_csv('dragon_data.csv')
df.head(10)

TypeError: 'NoneType' object is not subscriptable

## Basic overview of the data

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)

## 2- Missing values

In [ ]:
print("\n--- MISSING VALUES ---")
print(df.isnull().sum())


--- MISSING VALUES ---
SPC     0
AGE     0
MASS    0
WSP     0
HID     0
SPD     0
FHO     0
dtype: int64


## 3- Duplicate rows

In [ ]:
print("\n--- DUPLICATE ROWS ---")
print(f"Number of duplicate rows: {df.duplicated().sum()}")


--- DUPLICATE ROWS ---
Number of duplicate rows: 0


## 4- Typos in the data (column)

In [ ]:
print("\n--- SPECIES VALUES (check for typos) ---")
print(df['SPC'].value_counts())
# Any value that appears very few times is likely a typo
print("\nRare species values (possible typos):")
print(df['SPC'].value_counts()[df['SPC'].value_counts() < 5])



--- SPECIES VALUES (check for typos) ---
SPC
Hydra      214
Wyvern     183
Dragon     102
Wyvernn      1
Name: count, dtype: int64

Rare species values (possible typos):
SPC
Wyvernn    1
Name: count, dtype: int64


## 5- Let check impossible values (numeric column)

In [ ]:
print("\n--- NUMERIC SUMMARY ---")
print(df.describe().round(2))

print("\n--- INVALID VALUES CHECK ---")

# AGE: must be positive
invalid_age = df[df['AGE'] <= 0]
print(f"\nRows with AGE <= 0: {len(invalid_age)}")
if len(invalid_age) > 0:
    print(invalid_age)

# MASS: must be positive
invalid_mass = df[df['MASS'] <= 0]
print(f"\nRows with MASS <= 0: {len(invalid_mass)}")
if len(invalid_mass) > 0:
    print(invalid_mass)

# WSP: must be positive
invalid_wsp = df[df['WSP'] <= 0]
print(f"\nRows with WSP <= 0: {len(invalid_wsp)}")
if len(invalid_wsp) > 0:
    print(invalid_wsp)

# FHO: must be positive (can't have negative temperature in context)
invalid_fho = df[df['FHO'] <= 0]
print(f"\nRows with FHO <= 0: {len(invalid_fho)}")
if len(invalid_fho) > 0:
    print(invalid_fho)

# SPD: must be positive
invalid_spd = df[df['SPD'] <= 0]
print(f"\nRows with SPD <= 0: {len(invalid_spd)}")
if len(invalid_spd) > 0:
    print(invalid_spd)

# HID: must be positive
invalid_hid = df[df['HID'] <= 0]
print(f"\nRows with HID <= 0: {len(invalid_hid)}")
if len(invalid_hid) > 0:
    print(invalid_hid)


--- NUMERIC SUMMARY ---
           AGE        MASS    WSP     HID        SPD      FHO
count   500.00      500.00  500.0  500.00     500.00   500.00
mean    461.47   309248.69   18.2    3.00   24157.12   894.24
std     643.25   751723.85   11.5    1.17   26568.85   350.48
min    -481.00       17.00    1.0    1.01     600.00   100.00
25%     105.50    19392.25   10.4    1.92    7665.00   664.00
50%     240.00    79675.00   15.6    2.94   13890.00   871.50
75%     585.00   282531.50   24.2    4.04   29550.00  1109.50
max    5407.00  9729833.00   73.6    5.00  161920.00  2358.00

--- INVALID VALUES CHECK ---

Rows with AGE <= 0: 1
        SPC  AGE    MASS   WSP  HID    SPD   FHO
328  Wyvern -481  223733  21.9  1.4  26280  1276

Rows with MASS <= 0: 0

Rows with WSP <= 0: 0

Rows with FHO <= 0: 0

Rows with SPD <= 0: 0

Rows with HID <= 0: 0


## 6- Let check the outliers

In [ ]:
print("\n--- OUTLIERS (IQR method) ---")
numeric_cols = ['AGE', 'MASS', 'WSP', 'SPD', 'HID', 'FHO']

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers  (expected range: {lower:.1f} to {upper:.1f})")


--- OUTLIERS (IQR method) ---
AGE: 37 outliers  (expected range: -613.8 to 1304.2)
MASS: 57 outliers  (expected range: -375316.6 to 677240.4)
WSP: 16 outliers  (expected range: -10.3 to 44.9)
SPD: 48 outliers  (expected range: -25162.5 to 62377.5)
HID: 0 outliers  (expected range: -1.3 to 7.2)
FHO: 8 outliers  (expected range: -4.2 to 1777.8)


## 7- let summarize

In [ ]:
print("\n" + "="*50)
print("SUMMARY OF ERRORS FOUND")
print("="*50)
total_issues = (
    df.isnull().sum().sum() +
    df.duplicated().sum() +
    len(invalid_age) +
    len(invalid_mass) +
    len(invalid_wsp) +
    len(invalid_fho)
)
print(f"Total issues found: {total_issues}")
print(f"  Missing values   : {df.isnull().sum().sum()}")
print(f"  Duplicate rows   : {df.duplicated().sum()}")
print(f"  Invalid AGE      : {len(invalid_age)}")
print(f"  Invalid MASS     : {len(invalid_mass)}")
print(f"  Invalid WSP      : {len(invalid_wsp)}")
print(f"  Invalid FHO      : {len(invalid_fho)}")
print(f"  Species typos    : check value_counts() above")


SUMMARY OF ERRORS FOUND
Total issues found: 1
  Missing values   : 0
  Duplicate rows   : 0
  Invalid AGE      : 1
  Invalid MASS     : 0
  Invalid WSP      : 0
  Invalid FHO      : 0
  Species typos    : check value_counts() above


## 9- Load clean data

In [ ]:
df['SPC'] = df['SPC'].replace('Wyvernn', 'Wyvern')
df = df[df['AGE'] > 0].reset_index(drop=True)

train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['SPC'])
train = train.reset_index(drop=True)
test  = test.reset_index(drop=True)
y_tr  = train['FHO']
y_te  = test['FHO']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def r2_rmse(y_true, y_pred):
    return r2_score(y_true, y_pred), np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate(name, Xtr, Xte, verbose=True):
    lr = LinearRegression().fit(Xtr, y_tr)
    r2_tr, rmse_tr = r2_rmse(y_tr, lr.predict(Xtr))
    r2_te, rmse_te = r2_rmse(y_te, lr.predict(Xte))
    cv = cross_val_score(LinearRegression(), Xtr, y_tr, cv=kf, scoring='r2').mean()
    if verbose:
        print(f"  {name:<45} Train={r2_tr:.4f}  Test={r2_te:.4f}  CV={cv:.4f}  RMSE={rmse_te:.1f}")
    return {'name': name, 'train': r2_tr, 'test': r2_te, 'cv': cv,
            'rmse': rmse_te, 'model': lr, 'Xtr': Xtr, 'Xte': Xte}



## 10- Baseline

In [ ]:
def base_features(data):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['WSP2']     = d['WSP'] ** 2
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    return pd.concat([d[['WSP','WSP2','log_MASS','log_AGE']].reset_index(drop=True),
                      dummies.reset_index(drop=True)], axis=1)

Xtr_base = base_features(train)
Xte_base = base_features(test)

results = {}
print("="*75)
print("BASELINE")
print("="*75)
results['FE4 (baseline)'] = evaluate('FE4 (baseline)', Xtr_base, Xte_base)

BASELINE
  FE4 (baseline)                                Train=0.5053  Test=0.6085  CV=0.4642  RMSE=220.1


## 11- Alternative single variable transformation

In [ ]:
print("\n" + "="*70)
print("PART 1 – SINGLE VARIABLE REGRESSION")
print("="*70)

fig1, axes = plt.subplots(2, 2, figsize=(13, 10))
fig1.suptitle("Part 1 – Single-Variable Regression Models", fontsize=13, fontweight='bold')

single_results = {}

# ── 1a  WSP ───────────────────────────────────────────────────────────────────
ax = axes[0, 0]
for spc, grp in train.groupby('SPC'):
    ax.scatter(grp['WSP'], grp['FHO'], alpha=0.5, s=20, color=COLORS[spc], label=spc)

X_tr = train[['WSP']]; y_tr = train['FHO']
X_te = test[['WSP']];  y_te = test['FHO']
lr_wsp = LinearRegression().fit(X_tr, y_tr)
r2_tr, rmse_tr = r2_rmse(lr_wsp.predict(X_tr), y_tr)
r2_te, rmse_te = r2_rmse(lr_wsp.predict(X_te), y_te)
single_results['WSP'] = (r2_tr, r2_te, rmse_tr, rmse_te)

xline = np.linspace(df['WSP'].min(), df['WSP'].max(), 200).reshape(-1,1)
ax.plot(xline, lr_wsp.predict(xline), 'k--', lw=2, label='Fit')
ax.set_xlabel('Wingspan (m)'); ax.set_ylabel('FHO (°C)')
ax.set_title(f'(a) WSP  –  Train R²={r2_tr:.3f}  |  Test R²={r2_te:.3f}')
ax.legend(fontsize=8)
print(f"\nWSP: coeff={lr_wsp.coef_[0]:.3f}, intercept={lr_wsp.intercept_:.1f}")
print(f"     Train R²={r2_tr:.4f}, RMSE={rmse_tr:.1f}  |  Test R²={r2_te:.4f}, RMSE={rmse_te:.1f}")

# ── 1b  MASS (log scale) ──────────────────────────────────────────────────────
ax = axes[0, 1]
for spc, grp in train.groupby('SPC'):
    ax.scatter(grp['MASS'], grp['FHO'], alpha=0.5, s=20, color=COLORS[spc], label=spc)

X_tr = train[['MASS']]; X_te = test[['MASS']]
lr_mass = LinearRegression().fit(X_tr, y_tr)
r2_tr, rmse_tr = r2_rmse(lr_mass.predict(X_tr), y_tr)
r2_te, rmse_te = r2_rmse(lr_mass.predict(X_te), y_te)
single_results['MASS'] = (r2_tr, r2_te, rmse_tr, rmse_te)

xline = np.linspace(df['MASS'].min(), df['MASS'].max(), 200).reshape(-1,1)
ax.plot(xline, lr_mass.predict(xline), 'k--', lw=2, label='Fit')
ax.set_xscale('log'); ax.set_xlabel('Mass (metric tons, log scale)'); ax.set_ylabel('FHO (°C)')
ax.set_title(f'(b) MASS  –  Train R²={r2_tr:.3f}  |  Test R²={r2_te:.3f}')
ax.legend(fontsize=8)
print(f"\nMASS: coeff={lr_mass.coef_[0]:.6f}, intercept={lr_mass.intercept_:.1f}")
print(f"      Train R²={r2_tr:.4f}, RMSE={rmse_tr:.1f}  |  Test R²={r2_te:.4f}, RMSE={rmse_te:.1f}")

# ── 1c  AGE ───────────────────────────────────────────────────────────────────
ax = axes[1, 0]
for spc, grp in train.groupby('SPC'):
    ax.scatter(grp['AGE'], grp['FHO'], alpha=0.5, s=20, color=COLORS[spc], label=spc)

X_tr = train[['AGE']]; X_te = test[['AGE']]
lr_age = LinearRegression().fit(X_tr, y_tr)
r2_tr, rmse_tr = r2_rmse(lr_age.predict(X_tr), y_tr)
r2_te, rmse_te = r2_rmse(lr_age.predict(X_te), y_te)
single_results['AGE'] = (r2_tr, r2_te, rmse_tr, rmse_te)

xline = np.linspace(df['AGE'].min(), df['AGE'].max(), 200).reshape(-1,1)
ax.plot(xline, lr_age.predict(xline), 'k--', lw=2, label='Fit')
ax.set_xlabel('Age (years)'); ax.set_ylabel('FHO (°C)')
ax.set_title(f'(c) AGE  –  Train R²={r2_tr:.3f}  |  Test R²={r2_te:.3f}')
ax.legend(fontsize=8)
print(f"\nAGE: coeff={lr_age.coef_[0]:.4f}, intercept={lr_age.intercept_:.1f}")
print(f"     Train R²={r2_tr:.4f}, RMSE={rmse_tr:.1f}  |  Test R²={r2_te:.4f}, RMSE={rmse_te:.1f}")

# ── 1d  SPC (one-hot, drop one) ───────────────────────────────────────────────
ax = axes[1, 1]
spc_dummies_tr = pd.get_dummies(train['SPC'], drop_first=True)
spc_dummies_te = pd.get_dummies(test['SPC'],  drop_first=True)
# Align columns
spc_dummies_te = spc_dummies_te.reindex(columns=spc_dummies_tr.columns, fill_value=0)

lr_spc = LinearRegression().fit(spc_dummies_tr, train['FHO'])
r2_tr, rmse_tr = r2_rmse(lr_spc.predict(spc_dummies_tr), train['FHO'])
r2_te, rmse_te = r2_rmse(lr_spc.predict(spc_dummies_te), test['FHO'])
single_results['SPC'] = (r2_tr, r2_te, rmse_tr, rmse_te)

species_means = df.groupby('SPC')['FHO'].mean().sort_values()
colors_bar = [COLORS[s] for s in species_means.index]
bars = ax.bar(species_means.index, species_means.values, color=colors_bar, edgecolor='k', linewidth=0.7)
for bar, val in zip(bars, species_means.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+5, f'{val:.0f}°C', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Species'); ax.set_ylabel('Mean FHO (°C)')
ax.set_title(f'(d) SPC  –  Train R²={r2_tr:.3f}  |  Test R²={r2_te:.3f}')
print(f"\nSPC: intercept={lr_spc.intercept_:.1f}")
for col, coef in zip(spc_dummies_tr.columns, lr_spc.coef_):
    print(f"     {col}: coeff={coef:.1f}")
print(f"     Train R²={r2_tr:.4f}, RMSE={rmse_tr:.1f}  |  Test R²={r2_te:.4f}, RMSE={rmse_te:.1f}")

plt.tight_layout()
plt.savefig('fig1_single_variable.png', bbox_inches='tight')
plt.close()
print("\nFigure 1 saved.")

# Summary table
print("\nSingle-variable summary:")
print(f"{'Variable':<10} {'Train R²':>10} {'Test R²':>10} {'Train RMSE':>12} {'Test RMSE':>12}")
for var, (r2_tr, r2_te, rmse_tr, rmse_te) in single_results.items():
    print(f"{var:<10} {r2_tr:>10.4f} {r2_te:>10.4f} {rmse_tr:>12.1f} {rmse_te:>12.1f}")


PART 1 – SINGLE VARIABLE REGRESSION

WSP: coeff=21.979, intercept=496.9
     Train R²=-0.0750, RMSE=251.5  |  Test R²=0.4021, RMSE=222.0

MASS: coeff=0.000332, intercept=796.1
      Train R²=-0.8394, RMSE=281.2  |  Test R²=0.2932, RMSE=310.7

AGE: coeff=0.3788, intercept=721.5
     Train R²=-0.3496, RMSE=264.8  |  Test R²=0.3617, RMSE=234.0

SPC: intercept=1093.8
     Hydra: coeff=-322.6
     Wyvern: coeff=-186.8
     Train R²=-6.3140, RMSE=327.7  |  Test R²=-5.1137, RMSE=297.4

Figure 1 saved.

Single-variable summary:
Variable     Train R²    Test R²   Train RMSE    Test RMSE
WSP           -0.0750     0.4021        251.5        222.0
MASS          -0.8394     0.2932        281.2        310.7
AGE           -0.3496     0.3617        264.8        234.0
SPC           -6.3140    -5.1137        327.7        297.4


In [ ]:
print("\n" + "="*75)
print("BLOCK 1 – ALTERNATIVE TRANSFORMATIONS")
print("="*75)

def make_transformed(data, extra_cols, include_spc=True):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['log_SPD']  = np.log(d['SPD'])
    d['sqrt_MASS'] = np.sqrt(d['MASS'])
    d['sqrt_AGE']  = np.sqrt(d['AGE'])
    d['sqrt_WSP']  = np.sqrt(d['WSP'])
    d['log_WSP']   = np.log(d['WSP'])
    d['WSP2']      = d['WSP'] ** 2
    d['log_MASS2'] = d['log_MASS'] ** 2
    d['log_AGE2']  = d['log_AGE'] ** 2
    cols = extra_cols.copy()
    if include_spc:
        dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
        d = pd.concat([d, dummies], axis=1)
        cols += [c for c in ['Hydra','Wyvern'] if c in d.columns]
    return d[cols]

# Test individual transforms against the baseline
transforms = {
    'log(WSP) + log(MASS) + log(AGE) + SPC':
        ['log_WSP','log_MASS','log_AGE'],
    'sqrt(WSP) + log(MASS) + log(AGE) + SPC':
        ['sqrt_WSP','log_MASS','log_AGE'],
    'WSP + WSP² + log(MASS) + log(AGE)² + SPC':
        ['WSP','WSP2','log_MASS','log_AGE','log_AGE2'],
    'WSP + WSP² + log(MASS)² + log(AGE) + SPC':
        ['WSP','WSP2','log_MASS','log_MASS2','log_AGE'],
    'WSP + sqrt(MASS) + log(AGE) + SPC':
        ['WSP','sqrt_MASS','log_AGE'],
    'WSP + WSP² + log(MASS) + sqrt(AGE) + SPC':
        ['WSP','WSP2','log_MASS','sqrt_AGE'],
}

for name, cols in transforms.items():
    Xtr = make_transformed(train, cols)
    Xte = make_transformed(test,  cols)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    results[name] = evaluate(name, Xtr, Xte)


BLOCK 1 – ALTERNATIVE TRANSFORMATIONS
  log(WSP) + log(MASS) + log(AGE) + SPC         Train=0.4282  Test=0.5409  CV=0.3738  RMSE=238.4
  sqrt(WSP) + log(MASS) + log(AGE) + SPC        Train=0.4982  Test=0.6034  CV=0.4563  RMSE=221.5
  WSP + WSP² + log(MASS) + log(AGE)² + SPC      Train=0.5056  Test=0.6103  CV=0.4627  RMSE=219.6
  WSP + WSP² + log(MASS)² + log(AGE) + SPC      Train=0.5055  Test=0.6088  CV=0.4624  RMSE=220.0
  WSP + sqrt(MASS) + log(AGE) + SPC             Train=0.4958  Test=0.5911  CV=0.4550  RMSE=224.9
  WSP + WSP² + log(MASS) + sqrt(AGE) + SPC      Train=0.4920  Test=0.6069  CV=0.4497  RMSE=220.6


## Multiple variation regression

In [ ]:
print("\n" + "="*70)
print("PART 2 – MULTIPLE VARIABLE REGRESSION")
print("="*70)

def make_features(data, num_cols, include_spc=False):
    X = data[num_cols].copy()
    if include_spc:
        dummies = pd.get_dummies(data['SPC'], drop_first=True, dtype=float)
        X = pd.concat([X.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
    return X

multi_results = {}

def fit_eval(name, X_tr, X_te, y_tr, y_te):
    lr = LinearRegression().fit(X_tr, y_tr)
    r2_tr, rmse_tr = r2_rmse(lr.predict(X_tr), y_tr)
    r2_te, rmse_te = r2_rmse(lr.predict(X_te), y_te)
    # Cross-val on train
    cv_r2 = cross_val_score(LinearRegression(), X_tr, y_tr, cv=5, scoring='r2').mean()
    multi_results[name] = (r2_tr, r2_te, rmse_tr, rmse_te, cv_r2, lr)
    print(f"\nModel: {name}")
    print(f"  Train R²={r2_tr:.4f}, Test R²={r2_te:.4f}, 5-CV R²={cv_r2:.4f}, RMSE(test)={rmse_te:.1f}")
    return lr

y_tr = train['FHO']; y_te = test['FHO']

# M1: WSP + MASS
Xtr = make_features(train, ['WSP','MASS']); Xte = make_features(test, ['WSP','MASS'])
m1 = fit_eval('M1: WSP+MASS', Xtr, Xte, y_tr, y_te)

# M2: WSP + MASS + AGE
Xtr = make_features(train, ['WSP','MASS','AGE']); Xte = make_features(test, ['WSP','MASS','AGE'])
m2 = fit_eval('M2: WSP+MASS+AGE', Xtr, Xte, y_tr, y_te)

# M3: All numerics
Xtr = make_features(train, ['WSP','MASS','AGE','SPD','HID']); Xte = make_features(test, ['WSP','MASS','AGE','SPD','HID'])
m3 = fit_eval('M3: All numerics', Xtr, Xte, y_tr, y_te)

# M4: All numerics + SPC
Xtr = make_features(train, ['WSP','MASS','AGE','SPD','HID'], include_spc=True)
Xte = make_features(test,  ['WSP','MASS','AGE','SPD','HID'], include_spc=True)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
m4 = fit_eval('M4: All+SPC', Xtr, Xte, y_tr, y_te)

# M5: WSP + AGE + SPC
Xtr = make_features(train, ['WSP','AGE'], include_spc=True)
Xte = make_features(test,  ['WSP','AGE'], include_spc=True)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
m5 = fit_eval('M5: WSP+AGE+SPC', Xtr, Xte, y_tr, y_te)



PART 2 – MULTIPLE VARIABLE REGRESSION

Model: M1: WSP+MASS
  Train R²=-0.0692, Test R²=0.4397, 5-CV R²=0.4659, RMSE(test)=222.7

Model: M2: WSP+MASS+AGE
  Train R²=-0.0517, Test R²=0.4786, 5-CV R²=0.4693, RMSE(test)=235.4

Model: M3: All numerics
  Train R²=-0.0487, Test R²=0.4815, 5-CV R²=0.4548, RMSE(test)=233.8

Model: M4: All+SPC
  Train R²=-0.0313, Test R²=0.4870, 5-CV R²=0.4556, RMSE(test)=238.3

Model: M5: WSP+AGE+SPC
  Train R²=-0.0627, Test R²=0.4258, 5-CV R²=0.4604, RMSE(test)=220.3


## Feature Engineering (check early if we can have a good moddel)

In [ ]:
print("\n" + "="*70)
print("PART 6 – FEATURE ENGINEERING")
print("="*70)

def engineer(data):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['log_SPD']  = np.log(d['SPD'])
    d['WSP2']     = d['WSP'] ** 2
    d['WSP_x_logMASS'] = d['WSP'] * d['log_MASS']
    d['SPD_x_HID']     = d['SPD'] * d['HID']
    # SPC dummies
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    d = pd.concat([d, dummies], axis=1)
    return d

tr_e = engineer(train)
te_e = engineer(test)

# FE1: log transforms only
fe1_cols = ['WSP','log_MASS','log_AGE','log_SPD','HID']
Xtr = tr_e[fe1_cols]; Xte = te_e[fe1_cols]
m_fe1 = fit_eval('FE1: logs (WSP,logM,logA,logS,HID)', Xtr, Xte, y_tr, y_te)

# FE2: logs + SPC
fe2_cols = fe1_cols + ['Hydra','Wyvern']
# check available cols
fe2_cols = [c for c in fe2_cols if c in tr_e.columns]
Xtr = tr_e[fe2_cols]; Xte = te_e[fe2_cols]
m_fe2 = fit_eval('FE2: logs+SPC', Xtr, Xte, y_tr, y_te)

# FE3: logs + interactions + SPC
fe3_cols = fe1_cols + ['WSP2','WSP_x_logMASS','SPD_x_HID'] + [c for c in ['Hydra','Wyvern'] if c in tr_e.columns]
Xtr = tr_e[fe3_cols]; Xte = te_e[fe3_cols]
m_fe3 = fit_eval('FE3: logs+interactions+SPC', Xtr, Xte, y_tr, y_te)

# FE4: WSP + WSP2 + logMASS + logAGE + SPC  (parsimonious)
fe4_cols = ['WSP','WSP2','log_MASS','log_AGE'] + [c for c in ['Hydra','Wyvern'] if c in tr_e.columns]
Xtr = tr_e[fe4_cols]; Xte = te_e[fe4_cols]
m_fe4 = fit_eval('FE4: WSP+WSP²+logM+logA+SPC', Xtr, Xte, y_tr, y_te)

# ── Identify best model ────────────────────────────────────────────────────────
best_name = max(multi_results, key=lambda k: multi_results[k][1])  # best test R²
print(f"\nBest model by test R²: {best_name}  ({multi_results[best_name][1]:.4f})")




PART 6 – FEATURE ENGINEERING

Model: FE1: logs (WSP,logM,logA,logS,HID)
  Train R²=0.0150, Test R²=0.4396, 5-CV R²=0.4758, RMSE(test)=220.7

Model: FE2: logs+SPC
  Train R²=0.0154, Test R²=0.4376, 5-CV R²=0.4719, RMSE(test)=221.3

Model: FE3: logs+interactions+SPC
  Train R²=0.0253, Test R²=0.4342, 5-CV R²=0.4649, RMSE(test)=221.8

Model: FE4: WSP+WSP²+logM+logA+SPC
  Train R²=0.0210, Test R²=0.4574, 5-CV R²=0.4805, RMSE(test)=220.1

Best model by test R²: M4: All+SPC  (0.4870)


## Model comparison Fig

In [ ]:
print("\n" + "="*70)
print("PART 3 – MODEL COMPARISON")
print("="*70)

names  = list(multi_results.keys())
tr_r2s = [multi_results[n][0] for n in names]
te_r2s = [multi_results[n][1] for n in names]
cv_r2s = [multi_results[n][4] for n in names]

fig3, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(names))
w = 0.28
ax.bar(x - w, tr_r2s, w, label='Train R²',   color='#3498db', edgecolor='k', linewidth=0.5)
ax.bar(x,     te_r2s, w, label='Test R²',    color='#e74c3c', edgecolor='k', linewidth=0.5)
ax.bar(x + w, cv_r2s, w, label='5-CV R²',   color='#2ecc71', edgecolor='k', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('R²'); ax.set_title('Part 3 – Model Comparison: Train / Test / Cross-Val R²')
ax.legend(); ax.set_ylim(0, 1.05)
ax.axhline(0.9, color='grey', linestyle=':', linewidth=0.8, label='R²=0.9')
plt.tight_layout()
plt.savefig('fig3_model_comparison.png', bbox_inches='tight')
plt.close()
print("Figure 3 saved.")

# Full comparison table
print(f"\n{'Model':<42} {'Train R²':>9} {'Test R²':>9} {'5-CV R²':>9} {'RMSE(te)':>10}")
for n in names:
    r2_tr, r2_te, rmse_tr, rmse_te, cv, _ = multi_results[n]
    print(f"{n:<42} {r2_tr:>9.4f} {r2_te:>9.4f} {cv:>9.4f} {rmse_te:>10.1f}")


PART 3 – MODEL COMPARISON
Figure 3 saved.

Model                                       Train R²   Test R²   5-CV R²   RMSE(te)
M1: WSP+MASS                                 -0.0692    0.4397    0.4659      222.7
M2: WSP+MASS+AGE                             -0.0517    0.4786    0.4693      235.4
M3: All numerics                             -0.0487    0.4815    0.4548      233.8
M4: All+SPC                                  -0.0313    0.4870    0.4556      238.3
M5: WSP+AGE+SPC                              -0.0627    0.4258    0.4604      220.3
FE1: logs (WSP,logM,logA,logS,HID)            0.0150    0.4396    0.4758      220.7
FE2: logs+SPC                                 0.0154    0.4376    0.4719      221.3
FE3: logs+interactions+SPC                    0.0253    0.4342    0.4649      221.8
FE4: WSP+WSP²+logM+logA+SPC                   0.0210    0.4574    0.4805      220.1


## Interpretation we will use FE2

In [ ]:
print("\n" + "="*70)
print("PART 4 – INTERPRETABILITY  (Recommended model: FE4 WSP+WSP²+logM+logA+SPC)")
print("="*70)

fe4_cols_safe = [c for c in fe4_cols if c in tr_e.columns]
Xtr_fe4 = tr_e[fe4_cols_safe]
Xte_fe4 = te_e[fe4_cols_safe]

# Statsmodels for p-values & CI
Xtr_sm = sm.add_constant(Xtr_fe4)
ols = sm.OLS(y_tr, Xtr_sm).fit()
print(ols.summary())

# Coefficient plot
coefs = ols.params[1:]   # drop intercept
ci    = ols.conf_int().iloc[1:]
feat_labels = {'WSP':'Wingspan (m)', 'WSP2':'Wingspan² (m²)',
               'log_MASS':'log(Mass)', 'log_AGE':'log(Age)',
               'Hydra':'Species: Hydra', 'Wyvern':'Species: Wyvern'}

fig4, ax = plt.subplots(figsize=(8, 5))
sorted_idx = np.argsort(np.abs(coefs.values))
names_sorted = coefs.index[sorted_idx]
coef_sorted  = coefs.values[sorted_idx]
ci_lo = (coefs - ci[0]).values[sorted_idx]
ci_hi = (ci[1] - coefs).values[sorted_idx]

colors_c = ['#c0392b' if c > 0 else '#2980b9' for c in coef_sorted]
ax.barh(range(len(names_sorted)), coef_sorted, xerr=[ci_lo, ci_hi],
        color=colors_c, edgecolor='k', linewidth=0.5, capsize=4)
ax.set_yticks(range(len(names_sorted)))
ax.set_yticklabels([feat_labels.get(n, n) for n in names_sorted])
ax.axvline(0, color='k', linewidth=0.8)
ax.set_xlabel('Coefficient value')
ax.set_title('Part 4 – Coefficients with 95% CI (FE4 recommended model)')
plt.tight_layout()
plt.savefig('fig4_coefficients.png', bbox_inches='tight')
plt.close()
print("Figure 4 saved.")


PART 4 – INTERPRETABILITY  (Recommended model: FE4 WSP+WSP²+logM+logA+SPC)
                            OLS Regression Results                            
Dep. Variable:                    FHO   R-squared:                       0.505
Model:                            OLS   Adj. R-squared:                  0.498
Method:                 Least Squares   F-statistic:                     66.73
Date:                Sat, 02 May 2026   Prob (F-statistic):           6.20e-57
Time:                        17:46:07   Log-Likelihood:                -2762.3
No. Observations:                 399   AIC:                             5539.
Df Residuals:                     392   BIC:                             5567.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------

## Model Evaluation

In [ ]:
print("\n" + "="*70)
print("PART 5 – MODEL EVALUATION")
print("="*70)

best_lr = multi_results['FE4: WSP+WSP²+logM+logA+SPC'][5]
y_pred_tr = best_lr.predict(Xtr_fe4)
y_pred_te = best_lr.predict(Xte_fe4)
resid_tr  = y_tr.values - y_pred_tr
resid_te  = y_te.values - y_pred_te

# AIC / BIC via statsmodels
print(f"\nFE4 OLS AIC : {ols.aic:.1f}")
print(f"FE4 OLS BIC : {ols.bic:.1f}")
print(f"FE4 Train R²: {ols.rsquared:.4f}  Adj-R²: {ols.rsquared_adj:.4f}")

fig5, axes = plt.subplots(2, 3, figsize=(15, 9))
fig5.suptitle('Part 5 – Model Evaluation (FE2: logs+SPC)', fontsize=12, fontweight='bold')

# Residuals vs Fitted (train)
ax = axes[0, 0]
sc = ax.scatter(y_pred_tr, resid_tr, alpha=0.4, s=15,
                c=[{'Dragon':0,'Wyvern':1,'Hydra':2}[s] for s in train['SPC'].values],
                cmap='Set1')
ax.axhline(0, color='k', lw=1.2); ax.set_xlabel('Fitted FHO'); ax.set_ylabel('Residual')
ax.set_title('Residuals vs Fitted (train)')

# Residuals vs Fitted (test)
ax = axes[0, 1]
c_te = [{'Dragon':0,'Wyvern':1,'Hydra':2}[s] for s in test['SPC'].values]
ax.scatter(y_pred_te, resid_te, alpha=0.5, s=18, c=c_te, cmap='Set1')
ax.axhline(0, color='k', lw=1.2); ax.set_xlabel('Fitted FHO'); ax.set_ylabel('Residual')
ax.set_title('Residuals vs Fitted (test)')

# QQ plot (train residuals)
ax = axes[0, 2]
from scipy import stats
(osm, osr), (slope, intercept, r) = stats.probplot(resid_tr, dist='norm')
ax.scatter(osm, osr, alpha=0.4, s=12)
ax.plot(osm, slope*np.array(osm)+intercept, 'r-', lw=1.5)
ax.set_xlabel('Theoretical quantiles'); ax.set_ylabel('Sample quantiles')
ax.set_title('Normal Q-Q (train residuals)')

# Predicted vs Actual (test)
ax = axes[1, 0]
ax.scatter(y_te, y_pred_te, alpha=0.5, s=18, c=c_te, cmap='Set1')
lo, hi = min(y_te.min(), y_pred_te.min()), max(y_te.max(), y_pred_te.max())
ax.plot([lo,hi],[lo,hi],'k--', lw=1.5)
ax.set_xlabel('Actual FHO'); ax.set_ylabel('Predicted FHO')
ax.set_title(f'Pred vs Actual (test)  R²={r2_score(y_te, y_pred_te):.3f}')

# Residual distribution (test)
ax = axes[1, 1]
ax.hist(resid_te, bins=25, color='#3498db', edgecolor='k', linewidth=0.5)
ax.axvline(0, color='r', lw=1.5)
ax.set_xlabel('Residual'); ax.set_ylabel('Count')
ax.set_title(f'Residual distribution (test)\nMean={resid_te.mean():.1f}, SD={resid_te.std():.1f}')

# Cross-val R² across all models
ax = axes[1, 2]
all_models   = list(multi_results.keys())
all_cv_r2    = [multi_results[n][4] for n in all_models]
bar_colors   = ['#e74c3c' if n == 'FE2: logs+SPC' else '#95a5a6' for n in all_models]
ax.barh(range(len(all_models)), all_cv_r2, color=bar_colors, edgecolor='k', linewidth=0.5)
ax.set_yticks(range(len(all_models))); ax.set_yticklabels(all_models, fontsize=7)
ax.set_xlabel('5-Fold CV R²'); ax.set_title('Cross-val R² all models\n(red = recommended)')

plt.tight_layout()
plt.savefig('fig5_evaluation.png', bbox_inches='tight')
plt.close()
print("Figure 5 saved.")


PART 5 – MODEL EVALUATION

FE4 OLS AIC : 5538.6
FE4 OLS BIC : 5566.6
FE4 Train R²: 0.5053  Adj-R²: 0.4977
Figure 5 saved.


## Fig of feature Eng

In [ ]:
fig6, axes = plt.subplots(1, 3, figsize=(14, 4))
fig6.suptitle('Part 6 – Feature Engineering: Key Transformations', fontsize=12, fontweight='bold')

ax = axes[0]
for spc, grp in train.groupby('SPC'):
    ax.scatter(np.log(grp['MASS']), grp['FHO'], alpha=0.4, s=15, color=COLORS[spc], label=spc)
ax.set_xlabel('log(Mass)'); ax.set_ylabel('FHO (°C)'); ax.set_title('log(MASS) vs FHO')
ax.legend(fontsize=8)

ax = axes[1]
for spc, grp in train.groupby('SPC'):
    ax.scatter(np.log(grp['AGE']), grp['FHO'], alpha=0.4, s=15, color=COLORS[spc], label=spc)
ax.set_xlabel('log(Age)'); ax.set_ylabel('FHO (°C)'); ax.set_title('log(AGE) vs FHO')

ax = axes[2]
for spc, grp in train.groupby('SPC'):
    ax.scatter(grp['WSP'] * np.log(grp['MASS']), grp['FHO'], alpha=0.4, s=15, color=COLORS[spc], label=spc)
ax.set_xlabel('WSP × log(MASS)'); ax.set_ylabel('FHO (°C)'); ax.set_title('Interaction: WSP × log(MASS) vs FHO')

plt.tight_layout()
plt.savefig('fig6_feature_engineering.png', bbox_inches='tight')
plt.close()
print("Figure 6 saved.")

Figure 6 saved.


## correlation heatmap

In [ ]:
num_cols_all = ['WSP','MASS','AGE','SPD','HID','FHO']
corr = df[num_cols_all].corr()
fig_corr, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix (raw features + FHO)')
plt.tight_layout()
plt.savefig('fig0_correlations.png', bbox_inches='tight')
plt.close()
print("Correlation heatmap saved.")

print("\n\nAll done. Saved figures:")
for i, name in enumerate(['fig0_correlations','fig1_single_variable','fig3_model_comparison',
                           'fig4_coefficients','fig5_evaluation','fig6_feature_engineering']):
    print(f"  {name}.png")

Correlation heatmap saved.


All done. Saved figures:
  fig0_correlations.png
  fig1_single_variable.png
  fig3_model_comparison.png
  fig4_coefficients.png
  fig5_evaluation.png
  fig6_feature_engineering.png


##  Interactions

In [73]:
print("\n" + "="*75)
print("BLOCK 2 – SPECIES × NUMERIC INTERACTIONS")
print("="*75)

def spc_interact(data, num_cols):
    """Build features: numeric cols + SPC dummies + SPC×numeric interactions."""
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['log_SPD']  = np.log(d['SPD'])
    d['WSP2']     = d['WSP'] ** 2

    # dummies (Dragon = reference)
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    d = pd.concat([d, dummies.rename(columns=lambda x: x)], axis=1)
    spc_cols = [c for c in ['Hydra','Wyvern'] if c in d.columns]

    # interaction terms: each SPC dummy × each numeric
    for spc in spc_cols:
        for num in num_cols:
            d[f'{spc}_x_{num}'] = d[spc] * d[num]

    feat_cols = num_cols + spc_cols + [f'{s}_x_{n}' for s in spc_cols for n in num_cols]
    return d[feat_cols]

base_nums = ['WSP','WSP2','log_MASS','log_AGE']

# A: SPC × log(MASS) only
def feat_A(data):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['WSP2']     = d['WSP'] ** 2
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    d = pd.concat([d, dummies], axis=1)
    spc_cols = [c for c in ['Hydra','Wyvern'] if c in d.columns]
    for s in spc_cols:
        d[f'{s}_x_logM'] = d[s] * d['log_MASS']
    cols = ['WSP','WSP2','log_MASS','log_AGE'] + spc_cols + [f'{s}_x_logM' for s in spc_cols]
    return d[cols]

# B: SPC × log(AGE) only
def feat_B(data):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['WSP2']     = d['WSP'] ** 2
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    d = pd.concat([d, dummies], axis=1)
    spc_cols = [c for c in ['Hydra','Wyvern'] if c in d.columns]
    for s in spc_cols:
        d[f'{s}_x_logA'] = d[s] * d['log_AGE']
    cols = ['WSP','WSP2','log_MASS','log_AGE'] + spc_cols + [f'{s}_x_logA' for s in spc_cols]
    return d[cols]

# C: SPC × log(MASS) + SPC × log(AGE)
def feat_C(data):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['WSP2']     = d['WSP'] ** 2
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    d = pd.concat([d, dummies], axis=1)
    spc_cols = [c for c in ['Hydra','Wyvern'] if c in d.columns]
    for s in spc_cols:
        d[f'{s}_x_logM'] = d[s] * d['log_MASS']
        d[f'{s}_x_logA'] = d[s] * d['log_AGE']
    cols = (['WSP','WSP2','log_MASS','log_AGE'] + spc_cols
            + [f'{s}_x_logM' for s in spc_cols]
            + [f'{s}_x_logA' for s in spc_cols])
    return d[cols]
# D: SPC × WSP
def feat_D(data):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['WSP2']     = d['WSP'] ** 2
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    d = pd.concat([d, dummies], axis=1)
    spc_cols = [c for c in ['Hydra','Wyvern'] if c in d.columns]
    for s in spc_cols:
        d[f'{s}_x_WSP'] = d[s] * d['WSP']
    cols = (['WSP','WSP2','log_MASS','log_AGE'] + spc_cols
            + [f'{s}_x_WSP' for s in spc_cols])
    return d[cols]

# E: all SPC × numeric interactions
def feat_E(data):
    return spc_interact(data, base_nums)

for tag, fn in [('A: +SPC×logM',         feat_A),
                ('B: +SPC×logA',         feat_B),
                ('C: +SPC×logM+SPC×logA', feat_C),
                ('D: +SPC×WSP',          feat_D),
                ('E: +all SPC×numeric',  feat_E)]:
    Xtr = fn(train); Xte = fn(test)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    results[tag] = evaluate(tag, Xtr, Xte)



BLOCK 2 – SPECIES × NUMERIC INTERACTIONS
  A: +SPC×logM                                  Train=0.5090  Test=0.6029  CV=0.4636  RMSE=221.7
  B: +SPC×logA                                  Train=0.5090  Test=0.6027  CV=0.4635  RMSE=221.7
  C: +SPC×logM+SPC×logA                         Train=0.5140  Test=0.5889  CV=0.4554  RMSE=225.5
  D: +SPC×WSP                                   Train=0.5100  Test=0.6036  CV=0.4643  RMSE=221.5
  E: +all SPC×numeric                           Train=0.5259  Test=0.5820  CV=0.4625  RMSE=227.4


##  Biological and pysical interactions

In [74]:
print("\n" + "="*75)
print("BLOCK 3 – PHYSICAL / BIOLOGICAL INTERACTIONS")
print("="*75)

def phys_interact(data, extra_pairs, base_nums_list=None):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['log_SPD']  = np.log(d['SPD'])
    d['WSP2']     = d['WSP'] ** 2
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    d = pd.concat([d, dummies], axis=1)
    spc_cols = [c for c in ['Hydra','Wyvern'] if c in d.columns]
    for a, b in extra_pairs:
        d[f'{a}_x_{b}'] = d[a] * d[b]
    base = base_nums_list or ['WSP','WSP2','log_MASS','log_AGE']
    cols = base + spc_cols + [f'{a}_x_{b}' for a,b in extra_pairs]
    return d[cols]

phys_tests = {
    'logM × logA':         [('log_MASS','log_AGE')],
    'WSP × logM':          [('WSP','log_MASS')],
    'WSP × logA':          [('WSP','log_AGE')],
    'logSPD × logM':       [('log_SPD','log_MASS')],
    'HID × logM':          [('HID','log_MASS')],
    'HID × WSP':           [('HID','WSP')],
    'logM × logA + WSP×logM': [('log_MASS','log_AGE'),('WSP','log_MASS')],
}

for name, pairs in phys_tests.items():
    base = ['WSP','WSP2','log_MASS','log_AGE']
    needed = list({v for pair in pairs for v in pair if v not in base and v != 'HID'})
    full_base = base + ([v for v in needed if v not in base])
    if 'HID' in [v for pair in pairs for v in pair]:
        full_base.append('HID')
    Xtr = phys_interact(train, pairs, full_base)
    Xte = phys_interact(test,  pairs, full_base)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    results[f'Phys: {name}'] = evaluate(f'Phys: {name}', Xtr, Xte)




BLOCK 3 – PHYSICAL / BIOLOGICAL INTERACTIONS
  Phys: logM × logA                             Train=0.5055  Test=0.6095  CV=0.4627  RMSE=219.8
  Phys: WSP × logM                              Train=0.5056  Test=0.6105  CV=0.4588  RMSE=219.5
  Phys: WSP × logA                              Train=0.5063  Test=0.6081  CV=0.4625  RMSE=220.2
  Phys: logSPD × logM                           Train=0.5057  Test=0.6048  CV=0.4598  RMSE=221.1
  Phys: HID × logM                              Train=0.5066  Test=0.6071  CV=0.4616  RMSE=220.5
  Phys: HID × WSP                               Train=0.5055  Test=0.6076  CV=0.4590  RMSE=220.4
  Phys: logM × logA + WSP×logM                  Train=0.5057  Test=0.6113  CV=0.4565  RMSE=219.3


## Best combinations

In [75]:
print("\n" + "="*75)
print("BLOCK 4 – BEST COMBINATIONS (top picks from Blocks 1-3)")
print("="*75)

# Best C (SPC×logM + SPC×logA) is likely best; try adding logM×logA
def feat_best1(data):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['WSP2']     = d['WSP'] ** 2
    d['logM_x_logA'] = d['log_MASS'] * d['log_AGE']
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    d = pd.concat([d, dummies], axis=1)
    spc_cols = [c for c in ['Hydra','Wyvern'] if c in d.columns]
    for s in spc_cols:
        d[f'{s}_x_logM'] = d[s] * d['log_MASS']
        d[f'{s}_x_logA'] = d[s] * d['log_AGE']
    cols = (['WSP','WSP2','log_MASS','log_AGE','logM_x_logA'] + spc_cols
            + [f'{s}_x_logM' for s in spc_cols]
            + [f'{s}_x_logA' for s in spc_cols])
    return d[cols]

# Add WSP×logM on top of best1
def feat_best2(data):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['WSP2']     = d['WSP'] ** 2
    d['logM_x_logA'] = d['log_MASS'] * d['log_AGE']
    d['WSP_x_logM']  = d['WSP'] * d['log_MASS']
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    d = pd.concat([d, dummies], axis=1)
    spc_cols = [c for c in ['Hydra','Wyvern'] if c in d.columns]
    for s in spc_cols:
        d[f'{s}_x_logM'] = d[s] * d['log_MASS']
        d[f'{s}_x_logA'] = d[s] * d['log_AGE']
    cols = (['WSP','WSP2','log_MASS','log_AGE','logM_x_logA','WSP_x_logM'] + spc_cols
            + [f'{s}_x_logM' for s in spc_cols]
            + [f'{s}_x_logA' for s in spc_cols])
    return d[cols]

# Parsimonious: SPC×logM only + logM×logA
def feat_best3(data):
    d = data.copy()
    d['log_MASS'] = np.log(d['MASS'])
    d['log_AGE']  = np.log(d['AGE'])
    d['WSP2']     = d['WSP'] ** 2
    d['logM_x_logA'] = d['log_MASS'] * d['log_AGE']
    dummies = pd.get_dummies(d['SPC'], drop_first=True, dtype=float)
    d = pd.concat([d, dummies], axis=1)
    spc_cols = [c for c in ['Hydra','Wyvern'] if c in d.columns]
    for s in spc_cols:
        d[f'{s}_x_logM'] = d[s] * d['log_MASS']
    cols = (['WSP','WSP2','log_MASS','log_AGE','logM_x_logA'] + spc_cols
            + [f'{s}_x_logM' for s in spc_cols])
    return d[cols]

for tag, fn in [('BEST1: FE4+SPC×logM+SPC×logA+logM×logA', feat_best1),
                ('BEST2: BEST1+WSP×logM',                   feat_best2),
                ('BEST3: FE4+SPC×logM+logM×logA (parsimonious)', feat_best3)]:
    Xtr = fn(train); Xte = fn(test)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    results[tag] = evaluate(tag, Xtr, Xte)


BLOCK 4 – BEST COMBINATIONS (top picks from Blocks 1-3)
  BEST1: FE4+SPC×logM+SPC×logA+logM×logA        Train=0.5142  Test=0.5892  CV=0.4537  RMSE=225.5
  BEST2: BEST1+WSP×logM                         Train=0.5157  Test=0.5938  CV=0.4465  RMSE=224.2
  BEST3: FE4+SPC×logM+logM×logA (parsimonious)  Train=0.5094  Test=0.6040  CV=0.4627  RMSE=221.4


##  Let identifiers winners

In [76]:
print("\n" + "="*75)
print("FULL RANKING BY 5-CV R²")
print("="*75)
ranked = sorted(results.items(), key=lambda x: x[1]['cv'], reverse=True)
print(f"\n{'Model':<50} {'Train':>7} {'Test':>7} {'CV':>7} {'RMSE':>7}")
print("-"*80)
for name, r in ranked:
    marker = " <-- BEST" if name == ranked[0][0] else (
             " <-- baseline" if name == 'FE4 (baseline)' else "")
    print(f"{name:<50} {r['train']:>7.4f} {r['test']:>7.4f} {r['cv']:>7.4f} {r['rmse']:>7.1f}{marker}")

best_name, best_r = ranked[0]
print(f"\nBest model: {best_name}")
print(f"  CV R² improvement over baseline: "
      f"{best_r['cv'] - results['FE4 (baseline)']['cv']:.4f}")
print(f"  Test R² improvement over baseline: "
      f"{best_r['test'] - results['FE4 (baseline)']['test']:.4f}")


FULL RANKING BY 5-CV R²

Model                                                Train    Test      CV    RMSE
--------------------------------------------------------------------------------
D: +SPC×WSP                                         0.5100  0.6036  0.4643   221.5 <-- BEST
FE4 (baseline)                                      0.5053  0.6085  0.4642   220.1 <-- baseline
A: +SPC×logM                                        0.5090  0.6029  0.4636   221.7
B: +SPC×logA                                        0.5090  0.6027  0.4635   221.7
Phys: logM × logA                                   0.5055  0.6095  0.4627   219.8
WSP + WSP² + log(MASS) + log(AGE)² + SPC            0.5056  0.6103  0.4627   219.6
BEST3: FE4+SPC×logM+logM×logA (parsimonious)        0.5094  0.6040  0.4627   221.4
E: +all SPC×numeric                                 0.5259  0.5820  0.4625   227.4
Phys: WSP × logA                                    0.5063  0.6081  0.4625   220.2
WSP + WSP² + log(MASS)² + log(AGE) + SPC 

##  Figure scatter for best new interactions

In [77]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Feature Engineering: Interaction Term Exploration', fontsize=13, fontweight='bold')

tr_aug = train.copy()
tr_aug['log_MASS'] = np.log(tr_aug['MASS'])
tr_aug['log_AGE']  = np.log(tr_aug['AGE'])

# 1. log(MASS) vs FHO by species
ax = axes[0,0]
for spc, grp in tr_aug.groupby('SPC'):
    ax.scatter(grp['log_MASS'], grp['FHO'], alpha=0.4, s=15, color=COLORS[spc], label=spc)
    m = np.polyfit(grp['log_MASS'], grp['FHO'], 1)
    xs = np.linspace(grp['log_MASS'].min(), grp['log_MASS'].max(), 100)
    ax.plot(xs, np.polyval(m, xs), color=COLORS[spc], lw=1.5, linestyle='--')
ax.set_xlabel('log(MASS)'); ax.set_ylabel('FHO (°C)')
ax.set_title('log(MASS) vs FHO by species\n(different slopes → SPC×logM interaction)')
ax.legend(fontsize=8)

# 2. log(AGE) vs FHO by species
ax = axes[0,1]
for spc, grp in tr_aug.groupby('SPC'):
    ax.scatter(grp['log_AGE'], grp['FHO'], alpha=0.4, s=15, color=COLORS[spc], label=spc)
    m = np.polyfit(grp['log_AGE'], grp['FHO'], 1)
    xs = np.linspace(grp['log_AGE'].min(), grp['log_AGE'].max(), 100)
    ax.plot(xs, np.polyval(m, xs), color=COLORS[spc], lw=1.5, linestyle='--')
ax.set_xlabel('log(AGE)'); ax.set_ylabel('FHO (°C)')
ax.set_title('log(AGE) vs FHO by species\n(different slopes → SPC×logA interaction)')
ax.legend(fontsize=8)

# 3. log(MASS) × log(AGE) interaction
ax = axes[0,2]
tr_aug['logM_x_logA'] = tr_aug['log_MASS'] * tr_aug['log_AGE']
for spc, grp in tr_aug.groupby('SPC'):
    ax.scatter(grp['logM_x_logA'], grp['FHO'], alpha=0.4, s=15, color=COLORS[spc], label=spc)
ax.set_xlabel('log(MASS) × log(AGE)'); ax.set_ylabel('FHO (°C)')
ax.set_title('log(MASS)×log(AGE) interaction vs FHO')
ax.legend(fontsize=8)

# 4. Species-specific slopes for log(MASS) (coefficient plot per species)
ax = axes[1,0]
slope_data = {}
for spc, grp in tr_aug.groupby('SPC'):
    m = np.polyfit(grp['log_MASS'], grp['FHO'], 1)
    slope_data[spc] = m[0]
bars = ax.bar(slope_data.keys(), slope_data.values(),
              color=[COLORS[s] for s in slope_data.keys()],
              edgecolor='k', linewidth=0.7)
for bar, val in zip(bars, slope_data.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Slope (°C per unit log-mass)'); ax.set_xlabel('Species')
ax.set_title('Per-species slope of log(MASS) on FHO\n(heterogeneous slopes justify SPC×logM)')
# 5. Species-specific slopes for log(AGE)
ax = axes[1,1]
slope_data_age = {}
for spc, grp in tr_aug.groupby('SPC'):
    m = np.polyfit(grp['log_AGE'], grp['FHO'], 1)
    slope_data_age[spc] = m[0]
bars = ax.bar(slope_data_age.keys(), slope_data_age.values(),
              color=[COLORS[s] for s in slope_data_age.keys()],
              edgecolor='k', linewidth=0.7)
for bar, val in zip(bars, slope_data_age.values()):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height() + (3 if val >= 0 else -15),
            f'{val:.1f}', ha='center', va='bottom', fontsize=9)
ax.axhline(0, color='k', lw=0.8, linestyle='--')
ax.set_ylabel('Slope (°C per unit log-age)'); ax.set_xlabel('Species')
ax.set_title('Per-species slope of log(AGE) on FHO\n(sign differs across species!)')

# 6. CV R² comparison (top 10 models)
ax = axes[1,2]
top10 = ranked[:10]
names_short = [r[0][:35]+'…' if len(r[0])>35 else r[0] for r in top10]
cv_vals = [r[1]['cv'] for r in top10]
bar_colors = ['#e74c3c' if r[0] == best_name else
              '#f39c12' if r[0] == 'FE4 (baseline)' else '#95a5a6'
              for r in top10]
ax.barh(range(len(top10)), cv_vals, color=bar_colors, edgecolor='k', linewidth=0.5)
ax.set_yticks(range(len(top10))); ax.set_yticklabels(names_short[::-1][::-1], fontsize=7)
ax.set_xlabel('5-fold CV R²')
ax.set_title('Top 10 models by CV R²\n(red=best, orange=FE4 baseline)')
ax.axvline(results['FE4 (baseline)']['cv'], color='#f39c12', linestyle='--', lw=1.2)

plt.tight_layout()
plt.savefig('fig7_fe_interactions.png', bbox_inches='tight')
plt.close()
print("\nFigure 7 saved.")


Figure 7 saved.


## Figure on coefficient plot for best model vs FE4 baseline

In [78]:
# Run statsmodels on best model
best_fn_map = {
    'BEST1: FE4+SPC×logM+SPC×logA+logM×logA': feat_best1,
    'BEST3: FE4+SPC×logM+logM×logA (parsimonious)': feat_best3,
    'C: +SPC×logM+SPC×logA': feat_C,
}
fn_best = best_fn_map.get(best_name, feat_best1)
Xtr_best = best_r['Xtr']
ols_best = sm.OLS(y_tr, sm.add_constant(Xtr_best)).fit()
print("\n--- OLS summary: best new model ---")
print(ols_best.summary())

# Coefficient comparison figure
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 6))
fig2.suptitle('Coefficient Comparison: FE4 Baseline vs Best New Model', fontsize=12, fontweight='bold')

def coef_plot(ax, Xtr, y, title):
    ols = sm.OLS(y, sm.add_constant(Xtr)).fit()
    coefs = ols.params[1:]
    ci = ols.conf_int().iloc[1:]
    sorted_idx = np.argsort(np.abs(coefs.values))
    names_s = coefs.index[sorted_idx]
    coef_s = coefs.values[sorted_idx]
    ci_lo = (coefs - ci[0]).values[sorted_idx]
    ci_hi = (ci[1] - coefs).values[sorted_idx]
    sig = ols.pvalues[1:].values[sorted_idx] < 0.05
    colors_c = ['#c0392b' if c > 0 else '#2980b9' for c in coef_s]
    ax.barh(range(len(names_s)), coef_s, xerr=[ci_lo, ci_hi],
            color=colors_c, edgecolor='k', linewidth=0.5, capsize=3, alpha=0.8)
    for i, (name, s) in enumerate(zip(names_s, sig)):
        if s:
            ax.text(0.02, i, '*', transform=ax.get_yaxis_transform(),
                    ha='left', va='center', fontsize=12, color='k', fontweight='bold')
    ax.set_yticks(range(len(names_s))); ax.set_yticklabels(names_s, fontsize=8)
    ax.axvline(0, color='k', lw=0.8)
    ax.set_xlabel('Coefficient value')
    r2_tr, _ = r2_rmse(y, ols.fittedvalues)
    ax.set_title(f'{title}\nAdj R²={ols.rsquared_adj:.4f}, AIC={ols.aic:.1f}')
    return ols

ols_base = coef_plot(axes2[0], Xtr_base, y_tr, 'FE4 (baseline)')
ols_new  = coef_plot(axes2[1], Xtr_best, y_tr, f'Best new model\n({best_name[:40]}...)')

plt.tight_layout()
plt.savefig('fig8_coef_comparison.png', bbox_inches='tight')
plt.close()
print("Figure 8 saved.")


--- OLS summary: best new model ---
                            OLS Regression Results                            
Dep. Variable:                    FHO   R-squared:                       0.510
Model:                            OLS   Adj. R-squared:                  0.500
Method:                 Least Squares   F-statistic:                     50.75
Date:                Sat, 02 May 2026   Prob (F-statistic):           6.56e-56
Time:                        18:13:35   Log-Likelihood:                -2760.4
No. Observations:                 399   AIC:                             5539.
Df Residuals:                     390   BIC:                             5575.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const      

##  Residual comparison

In [79]:
fig3, axes3 = plt.subplots(2, 2, figsize=(13, 9))
fig3.suptitle('Residual Analysis: FE4 Baseline vs Best New Model', fontsize=12, fontweight='bold')

def resid_plots(axes_row, Xtr, Xte, label, color):
    lr = LinearRegression().fit(Xtr, y_tr)
    pred_te = lr.predict(Xte)
    resid_te = y_te.values - pred_te
    c_te = [{'Dragon':0,'Wyvern':1,'Hydra':2}[s] for s in test['SPC'].values]

    ax = axes_row[0]
    sc = ax.scatter(pred_te, resid_te, alpha=0.5, s=18, c=c_te, cmap='Set1')
    ax.axhline(0, color='k', lw=1.2)
    ax.set_xlabel('Fitted FHO (°C)'); ax.set_ylabel('Residual (°C)')
    r2_te = r2_score(y_te, pred_te)
    ax.set_title(f'{label}\nResiduals vs Fitted  (Test R²={r2_te:.4f})')

    ax = axes_row[1]
    ax.hist(resid_te, bins=25, color=color, edgecolor='k', linewidth=0.5, alpha=0.8)
    ax.axvline(0, color='r', lw=1.5)
    ax.set_xlabel('Residual (°C)'); ax.set_ylabel('Count')
    ax.set_title(f'{label}\nResiduals: mean={resid_te.mean():.1f}, SD={resid_te.std():.1f}')

resid_plots([axes3[0,0], axes3[0,1]], Xtr_base, Xte_base, 'FE4 baseline', '#3498db')
Xte_best = best_r['Xte']
resid_plots([axes3[1,0], axes3[1,1]], Xtr_best, Xte_best, f'Best new model', '#e74c3c')

plt.tight_layout()
plt.savefig('fig9_residuals_comparison.png', bbox_inches='tight')
plt.close()
print("Figure 9 saved.")

Figure 9 saved.
